# E04 · Align taxonomies across code systems

**Outcome:** Select a mapping relation, preserve its direction and scope, and keep proposals separate from accepted mappings.

**Time:** about 55 minutes. Run cells in order. This is the executed solution edition.

SKOS alignment relates concepts in different schemes. exactMatch is symmetric and transitive, so one mistaken link can spread through a network. closeMatch is symmetric but not transitive. broadMatch points from a narrower source concept to a broader target; narrowMatch reverses that direction. relatedMatch expresses an associative mapping without hierarchy. None of these relations means owl:sameAs, owl:equivalentClass or a patient diagnosis.

Our accepted broadMatch links place source code tokens into local reporting families. A local alias with exactly the same scope as source token 493 has an exactMatch to that token. The proposed link from source token 493 to ICD-10-CM J45 is represented as a review record. Reifying a statement does not assert its triple. That distinction lets a catalog display the proposal without allowing the query layer to treat it as approved.

The dataset is historical ICD-9-CM. The April 2026 ICD-10-CM vocabulary and January 2025 ICD-11 MMS references are separate releases for comparison. ICD-20 is not the WHO revision used here; ICD-11 is the current revision family. The three ICD-11 references are verified against WHO's release. A crosswalk may be approximate, one-to-many or require clinical context absent from a row. WHO cautions against treating crosswalks as automatic conversion. Do not replace a source diagnosis with a later code on the strength of a similar label.

For each mapping record the two schemes, versions, source and target IRIs, relation direction, rationale, evidence, reviewer role, status and intended use. Evaluate precision on reviewed accepted links, coverage against the source scope, and unresolved cases separately. A confidence score is not an approval.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Separate proposed from accepted statements

In [2]:
v=vocabulary()
assert (CODE['493'],SKOS.closeMatch,CM.J45) not in v
proposal=list(v.triples((LOCAL.proposal493,None,None)))
display(proposal)
display(list(query(v,'SELECT ?code ?family WHERE { ?code skos:broadMatch ?family } ORDER BY ?code')))

[(rdflib.term.URIRef('https://example.org/health/reporting/proposal493'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#Statement')),
 (rdflib.term.URIRef('https://example.org/health/reporting/proposal493'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#subject'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/493')),
 (rdflib.term.URIRef('https://example.org/health/reporting/proposal493'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate'),
  rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#closeMatch')),
 (rdflib.term.URIRef('https://example.org/health/reporting/proposal493'),
  rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#object'),
  rdflib.term.URIRef('https://example.org/health/icd10cm/2026-04/J45')),
 (rdflib.term.URIRef('https://example.org/health/reporting/proposal493'),
  rdflib.term.URIRef('https://exam

[(rdflib.term.URIRef('https://example.org/health/icd9-source/250.02'),
  rdflib.term.URIRef('https://example.org/health/reporting/endocrine')),
 (rdflib.term.URIRef('https://example.org/health/icd9-source/428'),
  rdflib.term.URIRef('https://example.org/health/reporting/cardiac')),
 (rdflib.term.URIRef('https://example.org/health/icd9-source/493'),
  rdflib.term.URIRef('https://example.org/health/reporting/respiratory'))]

## Read actual code reference files

In [3]:
import pandas as pd
display(pd.read_csv(ROOT/'data/icd10cm_terms.csv',dtype=str).loc[:,['code','label','release']])
display(pd.read_csv(ROOT/'data/icd11_references.csv',dtype=str).loc[:,['code','label','release']])
assert not any(str(o).startswith(str(CM)) for o in build_asserted().objects(None,EX.primaryCode))

,code,label,release
0,E11,Type 2 diabetes mellitus,2026-04-01
1,E11.9,Type 2 diabetes mellitus without complications,2026-04-01
2,I50,Heart failure,2026-04-01
3,I50.9,"Heart failure, unspecified",2026-04-01
4,J45,Asthma,2026-04-01
5,J45.909,"Unspecified asthma, uncomplicated",2026-04-01


,code,label,release
0,5A11,Type 2 diabetes mellitus,2025-01
1,BD10,Congestive heart failure,2025-01
2,CA23,Asthma,2025-01


## Your turn

Write the SPARQL query that returns source concepts and their local broader reporting families. Return the number of result rows.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = len(list(query(vocabulary(),'SELECT ?source ?target WHERE { ?source skos:broadMatch ?target }')))

In [5]:
learner_check(answer, lambda x: x == 3, 'The direction is source token -> broader reporting family.')

Exercise passed.
Out[0]: True


## Explain your model

What evidence would you need before changing closeMatch to exactMatch? Explain the consequence of transitivity.

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** Compare the proof premises, source scope and query contract described above; use your own words in a peer review.